In [1]:
import pandas as pd
import geopandas as gpd
import fiona
import glob
from functions import *

fiona.drvsupport.supported_drivers['KML'] = 'rw'

Load data

In [2]:
files = glob.glob("data_origin/geo_data/omi_perimeters/*.kml")

Merge .kml files into a single dataframe

In [3]:
df1 = gpd.GeoDataFrame(
    pd.concat([gpd.read_file(f, driver='KML') for f in files], ignore_index=True)
)

In [4]:
df = df1.copy()

Select relevant columns and clean names

In [5]:
# Select relevant columns
df = df[['Name', 'CODZONA', 'geometry']]

# Rename columns
df = df.rename(columns = {
    'Name' : 'mun_name',
    'CODZONA' : 'zone'
})

# Clean mun_name
df["mun_name"] = df["mun_name"].str.split("-").str[0].str.strip()

# Normalize mun_name and remove "39" (sobstitue for accents and apostrophes)
df["mun_name"] = df['mun_name'].apply(normalize_name).str.replace(r"39", "", regex = True)

Import ISTAT codes with area intersection

In [6]:
df_mun = gpd.read_file("datasets/geo_data/mun_perimeters.gpkg")

In [7]:
# Create unique zone identifier
df = df.reset_index(drop=True)
df["zone_id"] = df.index

df = df.to_crs(epsg=3857)

df_mun = df_mun.to_crs(epsg=3857)

# Intersect zones with municipalities
df_intersection = gpd.overlay(df, df_mun[["mun_istat", "geometry"]], how="intersection")

# Calculate intersection area
df_intersection["intersection_area"] = df_intersection.geometry.area

# Keep only the municipality with the largest overlap per zone
idx = df_intersection.groupby("zone_id")["intersection_area"].idxmax()
df_best = df_intersection.loc[idx, ["zone_id", "mun_istat"]]

# Merge ISTAT code back into df 
df = df.merge(df_best, on="zone_id", how="left")

Save data

In [8]:
df.to_file("datasets/geo_data/omi_zone_perimeters.gpkg", driver="GPKG", layer="zones")